# LMS-Based Schizophrenia Classification on Dataset 1 using Connectivity Features

This notebook implements an EEG schizophrenia classification pipeline on Dataset 1 using LMS-based adaptive filtering and connectivity features.

The EEG recordings are segmented into 5-second windows, filtered using an LMS procedure with the Fp1 channel as reference, and transformed into phase-based connectivity descriptors derived from PLV and PLI.

These connectivity features are then used for machine learning classification.

In [ ]:
import mne
from glob import glob
import numpy as np
from PyEMD import EMD
from scipy.signal import hilbert, lfilter, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.model_selection import KFold, cross_val_score
from sklearn.svm import SVC
from scipy.signal import butter, filtfilt

In [ ]:
def load_eeg(file_path):
    raw = mne.io.read_raw_edf(file_path, preload=True)
    return raw.get_data()
healthy_file_paths = sorted(glob(f"dataverse_files/h*.edf"))
schizo_file_paths = sorted(glob(f"dataverse_files/s*.edf"))
healthy_signals = [load_eeg(file) for file in healthy_file_paths]
schizo_signals = [load_eeg(file) for file in schizo_file_paths]

Extracting EDF parameters from /content/dataverse_files/h01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 231249  =      0.000 ...   924.996 secs...
Extracting EDF parameters from /content/dataverse_files/h02.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 227499  =      0.000 ...   909.996 secs...
Extracting EDF parameters from /content/dataverse_files/h03.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 227499  =      0.000 ...   909.996 secs...
Extracting EDF parameters from /content/dataverse_files/h04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 231249  =      0.000 ...   924.996 secs...
Extracting EDF parameters from /content/dataverse_files/h05.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 236249  

In [ ]:
def segment_signal(signal, segment_duration, fs):
    segment_length = int(segment_duration * fs)
    return [signal[:, i:i + segment_length] for i in range(0, signal.shape[1], segment_length)]

fs = 250
segment_duration = 5
healthy_segments = [segment_signal(sig, segment_duration, fs) for sig in healthy_signals]
schizo_segments = [segment_signal(sig, segment_duration, fs) for sig in schizo_signals]
healthy_segments = [seg for subj in healthy_segments for seg in subj]
schizo_segments = [seg for subj in schizo_segments for seg in subj]

In [ ]:
def extract_features_from_phases(phases_array):
    plv_feat = compute_plv(phases_array).flatten()
    pli_feat = compute_pli(phases_array).flatten()
    features = np.concatenate([plv_feat, pli_feat])
    return features


**LMS avec Reference**

In [ ]:
def lms_filter_1coeff(signal_1d, ref_1d, mu=0.005):
    n_times = len(signal_1d)
    output = np.zeros(n_times)
    w = 0.0
    for i in range(n_times):
        e = signal_1d[i] - w * ref_1d[i]
        w = w + 2*mu*e*ref_1d[i]
        output[i] = signal_1d[i] - e
    return output


In [ ]:
def lms_filter_fp1_allch(signal_2d, fp1_index=0, mu=0.005):
    signal_2d = np.asarray(signal_2d)
    n_channels, n_times = signal_2d.shape

    filtered = np.zeros_like(signal_2d)
    filtered[fp1_index, :] = signal_2d[fp1_index, :]
    for ch in range(n_channels):
        if ch == fp1_index:
            continue
        filtered[ch, :] = lms_filter_1coeff(
            signal_1d=signal_2d[ch, :],
            ref_1d=signal_2d[fp1_index, :],
            mu=mu
        )
    return filtered

In [ ]:
healthy_segments_filtered = Parallel(n_jobs=-1)(
    delayed(lms_filter_fp1_allch)(seg, fp1_index=0, mu=0.005)
    for seg in healthy_segments
)
schizo_segments_filtered = Parallel(n_jobs=-1)(
    delayed(lms_filter_fp1_allch)(seg, fp1_index=0, mu=0.005)
    for seg in schizo_segments
)

In [ ]:
def compute_phase_matrix(imfs_array):
    if imfs_array.size == 0:
        return np.empty((0, 0, 0))
    analytic_signal = hilbert(imfs_array, axis=2)
    phases = np.angle(analytic_signal)
    return phases

def compute_plv(phases):
    n_channels, n_imf, n_times = phases.shape
    plv_matrix = np.zeros((n_channels, n_channels, n_imf))
    for imf_idx in range(n_imf):
        for ch1 in range(n_channels):
            for ch2 in range(n_channels):
                phase_diff = phases[ch1, imf_idx, :] - phases[ch2, imf_idx, :]
                plv_matrix[ch1, ch2, imf_idx] = np.abs(np.mean(np.exp(1j * phase_diff)))
    return plv_matrix

def compute_pli(phases):
    n_channels, n_imf, n_times = phases.shape
    pli_matrix = np.zeros((n_channels, n_channels, n_imf))
    for imf_idx in range(n_imf):
        for ch1 in range(n_channels):
            for ch2 in range(n_channels):
                phase_diff = phases[ch1, imf_idx, :] - phases[ch2, imf_idx, :]
                pli_matrix[ch1, ch2, imf_idx] = np.abs(np.mean(np.sign(np.sin(phase_diff))))
    return pli_matrix

In [ ]:
def compute_connectivity_for_segment(filtered_segment):

    data_reshaped = filtered_segment[:, np.newaxis, :]

    phases = compute_phase_matrix(data_reshaped)

    plv = compute_plv(phases)
    pli = compute_pli(phases)
    return plv, pli

In [ ]:
healthy_connectivity = Parallel(n_jobs=-1)(
    delayed(compute_connectivity_for_segment)(seg)
    for seg in healthy_segments_filtered
)
schizo_connectivity = Parallel(n_jobs=-1)(
    delayed(compute_connectivity_for_segment)(seg)
    for seg in schizo_segments_filtered
)

In [ ]:
def extract_features_from_connectivity(conn_tuple):

    plv_3d, pli_3d = conn_tuple
    plv_mat = plv_3d[:, :, 0]
    pli_mat = pli_3d[:, :, 0]

    triu_idx = np.triu_indices(plv_mat.shape[0], k=1)
    plv_vals = plv_mat[triu_idx]
    pli_vals = pli_mat[triu_idx]

    feats = np.concatenate([plv_vals, pli_vals])
    return feats

In [ ]:
healthy_features = [extract_features_from_connectivity(conn) for conn in healthy_connectivity]
schizo_features  = [extract_features_from_connectivity(conn) for conn in schizo_connectivity]

X = np.vstack([healthy_features, schizo_features])
y = np.array([0]*len(healthy_features) + [1]*len(schizo_features))  # 0 = healthy, 1 = schizo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("=== RandomForest ===")
print(classification_report(y_test, y_pred))
print("Accuracy =", accuracy_score(y_test, y_pred))

=== RandomForest ===
              precision    recall  f1-score   support

           0       0.73      0.71      0.72       521
           1       0.77      0.78      0.77       634

    accuracy                           0.75      1155
   macro avg       0.75      0.75      0.75      1155
weighted avg       0.75      0.75      0.75      1155

Accuracy = 0.7497835497835498


In [ ]:
# KFold Cross Validation Random Forest

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='accuracy')
print(f"Random Forest - KFold CV scores: {rf_scores}")
print(f"Random Forest - Mean CV accuracy: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}")

Random Forest - KFold CV scores: [0.73939394 0.73593074 0.74285714 0.74199134 0.77835498]
Random Forest - Mean CV accuracy: 0.748 ± 0.016


In [ ]:
# KFold Cross Validation SVM

kf = KFold(n_splits=5, shuffle=True, random_state=42)

clf_svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
rf_scores = cross_val_score(clf_svm, X, y, cv=kf, scoring='accuracy')
print(f"SVM - KFold CV scores: {rf_scores}")
print(f"SVM - Mean CV accuracy: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}")

SVM - KFold CV scores: [0.67359307 0.66753247 0.68484848 0.65281385 0.70909091]
SVM - Mean CV accuracy: 0.678 ± 0.019
